# Title
Group G4_2: Nikolai Enger Hoff & Tanja Hasdal Gundersen \
Github link: https://github.com/NickNack124/INF201-group-deliverable-2

## Imports
Description

In [3]:
import json
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

## Task 1
Description

In [ ]:
# file = Path("../data/2026/library_cafe/POS_01/register_sales.log")
def file_reader(file: Path) -> tuple[str, str]:
    """Function that reads a file using using either utf-8 or latin1 as its encoding language"""
    try:
        encoder = 'utf-8'
        file_text_content = file.open(encoding=encoder).readlines()

    except UnicodeEncodeError:
        # In case utf-8 fails, we attempt to encode with latin1
        encoder = 'latin1'
        file_text_content = file.open(encoding=encoder).readlines()

    return (''.join(file_text_content), encoder)


def log_parser(file_details: tuple[str, str]) -> list[dict]:
    """Function that parses the output from the file_reader as a list of dictionaries"""
    pattern = r"""
        \[(?P<timestamp>\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\] # The time of the transaction
        \s+\[(?P<register_id>\w{3}_\d{2})\]                     # The register the transaction occured at
        \s+item=(?P<item>\w+)                                   # The item that was bought
        \s+price=(?P<price>\d+\.\d+)                            # The price of the item
        \s+(?P<currency>\w+)                                    # The currency the customer paid in
        \s+\((?P<status>\w+)\)                                  # The status of the transaction
    """
    # Verbose allows us to keep the pattern neat with a docstring and comments
    transactions = [m.groupdict() for m in re.finditer(pattern, file_details[0], re.VERBOSE)]

    # We then have to convert the price to a floating point number
    for record in transactions:
        record["price"] = float(record["price"])

    return transactions
#

#df = pd.DataFrame(log_parser(file_reader(Path("../data/2026/library_cafe/POS_01/register_sales.log"))))
#df

Description

## Task 2
Description

In [70]:
def file_finder(path: str | Path, suffix: str = '.log', relpath: bool = True) -> list[Path]:
    """Returns all the files of a specific suffix in a directory as paths in a list"""
    # First we convert the input to a valid path object
    dir = Path(path)
    if not dir.exists():
        dir = Path('..') / path
        # This is to make the code works as long as the input object at least shares a parent directory with the code
    if not relpath:
        # in case one desires the absolute paths instead
        dir = dir.resolve()

    # We then recursive dig through the directory and its subdirectories
    filelist = []
    for f in dir.iterdir():
        if f.is_dir():
            # We recursively call the function again, and extend filelist by the returned filelist
            filelist.extend(file_finder(str(f), suffix=suffix, relpath=relpath))

        elif f.is_file() and f.suffix == suffix:
            # If the file matches the suffix (default: '.log'), it gets appended
            filelist.append(f)

    return filelist


def metadata_reader(source: list | dict | str) -> set(str):    # type: ignore
    """'This function reads a json dictionary and returns a set of the metadata'"""
    # first we extract the json dictionary from the path
    if type(source) is str:
        source = json.load(file_finder(Path(source), suffix='.json')[0].open())
        # This does rely on the desired file being the first/only json-file in the directory

    # We standardize the iterable so the function can handle both lists and dictionaries
    if type(source) is dict:
        metadata = source.items()
    else:
        metadata = [(k, v) for k, v in enumerate(source)]

    active_registers = []
    for k, v in metadata:
        if type(v) in (list, dict):
            # extends the list with by appending the returned set
            active_registers.extend(metadata_reader(v))

        elif type(k) is str and k.lower() == "active" and type(v) is bool and v:
            # We make sure it actually has an active status, in case this dictionary were to ever include other booleans
            # This does still require the items ("id", str) and (str, bool) to be in source
            active_registers.append(source["id"])   # type: ignore (the logic gate already ensures this can only happen when source is a dict)

    # we added "type: ignore" at the def-line since it technically returns set(list[str]) and not set(str)
    return set(active_registers)
#

Description

## Task 3
Description

We want to remove glitch ouliers in the prices, as wall as turn the prices into a 1D array. We also want to calculate the refund ratio using the refund count and the sales count. 

In [ ]:


def remove_price_outliers(prices):
    ''' 
    Converts a list of prices to a 1D numpy array and removes outliers from a list of prices using 3 standard deviations from the mean. The outliers are replaced 
    with the median of the non-outlier prices.The function returns a 1D array.
    '''

    data = np.asarray(prices, dtype=float).flatten()

    mean = np.mean(data)
    std = np.std(data)
    outliers = np.abs(data - mean) > 3 * std

    median = np.median(data[~outliers])
    data[outliers] = median

    return data


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    '''
    Computes the refund ratio given the sales count and refund count. The addition of a small epsilon value is added to avoid zero-division errors. The function 
    returns a 1D array.
    '''
    
    epsilon = 1e-6
    refund_ratio = refund_count / (sales_count + epsilon)

    return refund_ratio



Description

## Task 4
Description

The transactions are converted into a pandas DataFrame, with time as the index, and sorts them cronologically. We group by item and resampple into 1 hour windows to sum the price, build a pivot table of revenue per item across success, refund and cancelled. We also compute the refund ratio per item by using the function form task 3.

In [ ]:


def create_dataframe(transactions):
    '''
    Creates a pandas DataFrame from a list of transactions. The function converts the
    'timestamp' column to a datetime object and sets it as the index of the DataFrame.
    '''

    df = pd.DataFrame(transactions)
    df['time'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp')
    df = df.sort_index()

    return df


def hourly_revenue(df):
    '''
    Computes the hourly revenue for each item in the DataFrame. The function groups the
    DataFrame by 'item' and resamples the 'price' column to hourly intervals,
    '''
    return df.groupby('item').resample('1h')['price'].sum()


def status_pivot(df):
    ''' 
    Creates a pivot table from the DataFrame with 'item' as the index, 'status' as the 
    columns, and the sum of 'price' as the values. The function fills any missing values with 0.
    '''
    return pd.pivot_table(
        df,
        values='price',
        index='item',
        columns='status',
        aggfunc='sum',
        fill_value=0
    )


def refund_ratio_per_item(df):
    '''
    Computes the refund ratio for each item in the DataFrame. The function first creates a pivot table 
    using the `status_pivot` function, then extracts the sales and refund counts for each item. Finally, 
    it computes the refund ratio using the `compute_refund_ratio` function and returns a 1D array of refund ratios.
    '''
    pivot = status_pivot(df)
    sales = pivot['SUCCESS']
    refunds = pivot['REFUND']
    
    return compute_refund_ratio(
        np.asarray(sales.values), np.asarray(refunds.values)
    )



Description

## Task 5
Description

Description